# Exploring a MongoDB workload

Use `dbrtoolkit` to **run queries, inspect plans, and handle workload failures**.
Run the cells in order with the project's Python kernel. This notebook expects
MongoDB at `127.0.0.1:27018` with the `yelp.business` dataset.

All examples are read-only: no documents or indexes are changed. Result limits keep
outputs small, but sorting and grouping can still examine many documents.

| Call | What you get |
|---|---|
| `run(query)` | Duration and fully consumed row count |
| `run(query, collect_results=True)` | The same, plus documents |
| `explain(query)` | Raw plan; verbosity controls whether statistics are collected |
| `run_workload(...)` / `explain_workload(...)` | One result per entry, in input order |

In [1]:
from copy import deepcopy
from pprint import pprint

from IPython.display import Markdown, display
from pymongo import AsyncMongoClient

from dbrtoolkit.mongodb import (
    AsyncMongoDBExecutor,
    MongoAggregateQuery,
    MongoCollectionRef,
    MongoFindQuery,
)
from dbrtoolkit.workload import Workload, WorkloadEntry

URI = "mongodb://127.0.0.1:27018"
TIMEOUT_SECONDS = 15
business = MongoCollectionRef("yelp", "business")

# Check connectivity without modifying the database.
async with AsyncMongoClient(URI, serverSelectionTimeoutMS=3000) as client:
    await client.admin.command("ping")
print("Connected. Query execution will use yelp.business.")

Connected. Query execution will use yelp.business.


## 1. Describe queries before executing them

A query object is data, not a connection. This workload combines three find queries
and one aggregation against the same MongoDB backend. Pattern IDs are omitted
because these examples were written by hand.

In [2]:
summary_fields = {"_id": 0, "name": 1, "city": 1, "stars": 1, "review_count": 1}

city_query = MongoFindQuery(
    business,
    {"city": "Santa Barbara"},
    projection=summary_fields,
    limit=20,
)
popular_query = MongoFindQuery(
    business,
    {"is_open": 1, "stars": {"$gte": 4.0}, "review_count": {"$gte": 100}},
    projection=summary_fields,
    sort=(("review_count", -1),),
    limit=20,
)
restaurant_query = MongoFindQuery(
    business,
    {"city": "Tucson", "categories": "Restaurants"},
    projection=summary_fields,
    sort=(("stars", -1), ("review_count", -1)),
    limit=20,
)
city_totals = MongoAggregateQuery(
    business,
    [
        {"$match": {"state": "CA", "is_open": 1}},
        {
            "$group": {
                "_id": "$city",
                "business_count": {"$sum": 1},
                "average_stars": {"$avg": "$stars"},
            }
        },
        {"$sort": {"business_count": -1, "_id": 1}},
        {"$limit": 10},
    ],
)
labels = ["Santa Barbara", "Popular open businesses", "Tucson restaurants", "CA city totals"]
workload = Workload(
    [
        WorkloadEntry(city_query),
        WorkloadEntry(popular_query),
        WorkloadEntry(restaurant_query),
        WorkloadEntry(city_totals),
    ]
)
workload.validate()
print(f"{len(workload)} valid query descriptions; none executed yet.")

4 valid query descriptions; none executed yet.


## 2. Run one query: count only or retain documents

Both modes consume the **entire cursor**, including later batches. With collection
disabled, `documents` is `None`, not an empty list. Single-query failures raise.
The context manager closes the client created by `from_uri()`.

In [3]:
async with AsyncMongoDBExecutor.from_uri(URI, timeout_seconds=TIMEOUT_SECONDS) as executor:
    counted = await executor.run(city_query)
    collected = await executor.run(city_query, collect_results=True)

print(f"Count-only: {counted.row_count} rows; documents={counted.documents}")
print(f"Collected:  {collected.row_count} rows; showing the first 3")
pprint(collected.documents[:3], sort_dicts=False)
assert counted.documents is None
assert len(collected.documents) == collected.row_count

Count-only: 20 rows; documents=None
Collected:  20 rows; showing the first 3
[{'name': 'Abby Rappoport, LAC, CMQ',
  'city': 'Santa Barbara',
  'stars': 5.0,
  'review_count': 7},
 {'name': 'H&M', 'city': 'Santa Barbara', 'stars': 3.0, 'review_count': 24},
 {'name': 'Helena Avenue Bakery',
  'city': 'Santa Barbara',
  'stars': 4.0,
  'review_count': 389}]


## 3. Run the workload sequentially or concurrently

`concurrency=1` runs entries in order. Larger values bound in-flight queries;
returned results still align with the original entries, including duplicates.
The small helper below displays outcomes without dumping every document.

In [4]:
def show_results(names, results):
    lines = [
        "| Query | Status | Rows | Client ms |",
        "|---|---|---:|---:|",
    ]
    for name, result in zip(names, results):
        if result is None:
            lines.append(f"| {name} | not started | — | — |")
        else:
            rows = "—" if result.row_count is None else str(result.row_count)
            lines.append(f"| {name} | {result.status} | {rows} | {result.elapsed_ms:.2f} |")
    display(Markdown("\n".join(lines)))


for concurrency in (1, 4):
    async with AsyncMongoDBExecutor.from_uri(
        URI,
        concurrency=concurrency,
        continue_on_error=True,
        timeout_seconds=TIMEOUT_SECONDS,
    ) as executor:
        results = await executor.run_workload(workload)
    display(Markdown(f"**Concurrency: {concurrency}**"))
    show_results(labels, results)

**Concurrency: 1**

| Query | Status | Rows | Client ms |
|---|---|---:|---:|
| Santa Barbara | success | 20 | 9.46 |
| Popular open businesses | success | 20 | 87.68 |
| Tucson restaurants | success | 20 | 48.34 |
| CA city totals | success | 10 | 37.81 |

**Concurrency: 4**

| Query | Status | Rows | Client ms |
|---|---|---:|---:|
| Santa Barbara | success | 20 | 5.09 |
| Popular open businesses | success | 20 | 57.08 |
| Tucson restaurants | success | 20 | 55.36 |
| CA city totals | success | 10 | 46.35 |

These durations include client/network work and full cursor consumption. This is
an API demonstration, **not a controlled benchmark**: connection startup, cache
warming, run order and contention differ. Do not interpret these two runs as a
reliable speedup comparison.

## 4. Explain one query: plan versus execution statistics

`queryPlanner` requests planning information. `executionStats` also performs server
execution to collect statistics. Explain is a separate operation: it does not
return the query's documents, and its timing is not normal query latency.
MongoDB explain bypasses the plan cache.

The executor preserves the raw response. The extraction below is intentionally
limited to this find-query example; aggregation and sharded plans can have different shapes.

In [5]:
async with AsyncMongoDBExecutor.from_uri(URI, timeout_seconds=TIMEOUT_SECONDS) as executor:
    plan = await executor.explain(popular_query, verbosity="queryPlanner")
    measured_plan = await executor.explain(popular_query, verbosity="executionStats")

print("Winning plan (find query):")
pprint(plan.explain.get("queryPlanner", {}).get("winningPlan", {}), depth=4)
print("\nSelected server execution statistics:")
stats = measured_plan.explain.get("executionStats", {})
pprint(
    {
        key: stats.get(key)
        for key in (
            "nReturned",
            "executionTimeMillis",
            "totalDocsExamined",
            "totalKeysExamined",
        )
    }
)
print(f"Client-observed explain request: {measured_plan.elapsed_ms:.2f} ms")
assert measured_plan.documents is None

Winning plan (find query):
{'inputStage': {'inputStage': {'direction': 'forward',
                               'filter': {'$and': [...]},
                               'stage': 'COLLSCAN'},
                'limitAmount': 20,
                'memLimit': 104857600,
                'sortPattern': {'review_count': -1},
                'stage': 'SORT',
                'type': 'simple'},
 'isCached': False,
 'stage': 'PROJECTION_SIMPLE',
 'transformBy': {'_id': 0, 'city': 1, 'name': 1, 'review_count': 1, 'stars': 1}}

Selected server execution statistics:
{'executionTimeMillis': 89,
 'nReturned': 20,
 'totalDocsExamined': 150346,
 'totalKeysExamined': 0}
Client-observed explain request: 90.09 ms


## 5. Explain the entire workload

The same concurrency and error policy apply. Here we request plans only and show
the top-level response keys. Inspect `explanations[i].explain` for a full response.

In [6]:
async with AsyncMongoDBExecutor.from_uri(
    URI,
    concurrency=2,
    continue_on_error=True,
    timeout_seconds=TIMEOUT_SECONDS,
) as executor:
    explanations = await executor.explain_workload(workload, verbosity="queryPlanner")

show_results(labels, explanations)
for name, result in zip(labels, explanations):
    if result is not None and result.explain is not None:
        print(f"{name}: {', '.join(result.explain.keys())}")

| Query | Status | Rows | Client ms |
|---|---|---:|---:|
| Santa Barbara | success | — | 11.18 |
| Popular open businesses | success | — | 11.99 |
| Tucson restaurants | success | — | 0.61 |
| CA city totals | success | — | 1.46 |

Santa Barbara: explainVersion, queryPlanner, command, serverInfo, serverParameters, ok
Popular open businesses: explainVersion, queryPlanner, command, serverInfo, serverParameters, ok
Tucson restaurants: explainVersion, queryPlanner, command, serverInfo, serverParameters, ok
CA city totals: explainVersion, stages, serverInfo, serverParameters, command, ok


## 6. Make a failure explicit

Structural validation does not recognize every MongoDB operator. This deliberately
invalid stage passes local validation but is rejected by the server. We use
`concurrency=1` so stopping behaviour is deterministic.

A result's `error` retains the exception; `status` distinguishes `error` from
`timeout`. `None` means an entry never started. When concurrency is higher,
already-running queries finish even after scheduling stops.

In [7]:
invalid_query = MongoAggregateQuery(business, [{"$unknown": True}])
error_workload = Workload(
    [
        WorkloadEntry(city_query),
        WorkloadEntry(invalid_query),
        WorkloadEntry(restaurant_query),
    ]
)

for continue_on_error in (False, True):
    async with AsyncMongoDBExecutor.from_uri(
        URI,
        concurrency=1,
        continue_on_error=continue_on_error,
        timeout_seconds=TIMEOUT_SECONDS,
    ) as executor:
        outcomes = await executor.run_workload(error_workload)
    display(Markdown(f"**Continue on error: {continue_on_error}**"))
    show_results(["Valid find", "Deliberately invalid stage", "Another valid find"], outcomes)
    for result in outcomes:
        if result is not None and result.error is not None:
            print(type(result.error).__name__, str(result.error)[:180])
    assert outcomes[1].status == "error"
    if not continue_on_error:
        assert outcomes[2] is None

**Continue on error: False**

| Query | Status | Rows | Client ms |
|---|---|---:|---:|
| Valid find | success | 20 | 11.60 |
| Deliberately invalid stage | error | — | 0.75 |
| Another valid find | not started | — | — |

OperationFailure Unrecognized pipeline stage name: '$unknown', full error: {'ok': 0.0, 'errmsg': "Unrecognized pipeline stage name: '$unknown'", 'code': 40324, 'codeName': 'Location40324'}


**Continue on error: True**

| Query | Status | Rows | Client ms |
|---|---|---:|---:|
| Valid find | success | 20 | 11.55 |
| Deliberately invalid stage | error | — | 0.76 |
| Another valid find | success | 20 | 82.69 |

OperationFailure Unrecognized pipeline stage name: '$unknown', full error: {'ok': 0.0, 'errmsg': "Unrecognized pipeline stage name: '$unknown'", 'code': 40324, 'codeName': 'Location40324'}


## 7. Save a query and make an independent variant

Workload entries retain query references. Use `deepcopy()` before editing when you
want a separate variant. Queries and workloads can both be restored. The workload example uses a temporary
directory that is automatically removed afterward.

In [8]:
from pathlib import Path
from tempfile import TemporaryDirectory

variant = deepcopy(city_query)
variant.limit = 5
restored = MongoFindQuery.from_json(variant.to_json())

assert city_query.limit == 20
assert restored.fingerprint() == variant.fingerprint()
assert variant.fingerprint() != city_query.fingerprint()
print("Original limit:", city_query.limit, "| Variant limit:", variant.limit)
print("Variant fingerprint:", variant.fingerprint()[:16], "…")


with TemporaryDirectory() as folder:
    path = Path(folder) / "yelp.jsonl"
    workload.save(path)
    restored_workload = Workload.load(path)
    assert [e.query.fingerprint() for e in restored_workload] == [
        e.query.fingerprint() for e in workload
    ]
    print(f"Saved and restored {len(restored_workload)} workload entries")

Original limit: 20 | Variant limit: 5
Variant fingerprint: 1abd87d414da6b8c …


### Try next

- Change a city or threshold in an independent query variant, then run and explain it.
- Inspect `explanations[3].explain` to compare aggregation and find response shapes.
- Enable `collect_results=True` for `city_totals` to see its grouped rows.

Do not mutate queries while they are executing. A timeout is an incomplete attempt,
not a successful query with zero results. For API details, see the local
[executor guide](http://127.0.0.1:8000/executor/).